# Interactive Script: **Co-register Data Cube**

**Author:** Baturalp Arisoy<br>
**Contact:** baturalp.arisoy@uni-wuerzburg.de - Call me Batu :)

## Overview
This notebook guides the user through the essentials of co-registration of Sentinel-2 cube
1. [Auto Co-registration](#1-auto-co-registration) <br>
HIGHLY RECOMMENDED! The code ensures maximum amount of possible co-registration by scanning through the entire scene. 
2. [Custom ROI Co-Registration](#2-custom-roi-co-registration)
Select a polygon using the interactive leafmap by selecting a known stable area on the surface. This function is much faster than first function, however satellite time-series are complicated and can result removing many analysis worth scenes (scenes with very low cloud percentage). Therefore, the first method is always recommended to sustain the maximum amount of available scenes!
3. [Spectral Profile Generator](#3-spectral-profile-generator)

> **Note 1:**<br><br>
> This algorithm works the best if the study area is large enough. The larger the area, the better result to get, especially if the surface features are heterogeneous! <br><br>
> In case you are working on small patch but still wish to co-register, please generate initial data cube with much larger area and after co-registration, clip raster at **1_Initial_Data_Cube.ipynb, Chapter 6**.

> **Note 2:**<br><br>
> The algorithm performs much better if the clouds are masked. <br><br>Even if you want to skip Notebook 2 (Cloud Mask Data Cube generation), please generate your initial data cube (Notebook 1) with SCL masking (cloud_masking = True)

> **Note 3:** <br><br>
> Please generate longer time series to see the effect of the co-registration (look at the example animations at "interactive/animations" folder). The dates in the tiny data cubes we generated in the previous examples are already majorly aligned. <br><br>
> If you want to skip custom threshold based cloud masking **2_CloudMask_Data_Cube**, you can set **cloud_masking=True** during initial data cube generation.<br><br>

## 1. Auto Co-Registration

In [1]:
from stac2cube import coregister_cube
from stac2cube import show_coregistration_parameter_help

In [ ]:
show_coregistration_parameter_help()

> **Note:** <br><br>
> At the end of the process, you will get co-registration summary. <br> <br>
> If there are many excluded scenes with low cloud percentage or even no-cloud, please change the parameters:<br> <br>
> **a)** Increasing grid_size usually solves the problem, **b)** Happens, if the area is small, **c)** Happens if the surface patch is highly homogeneous (e.g. forest, snow) **d)** Try selecting first vegetation season

In [ ]:
out_ds = coregister_cube(
    input_path="./results/test.nc", 
    grid_size=7, 
    max_cc=5,
    time_period= ["2023-03-01", "2023-12-31"],
    min_reliability_keep=10.0,
    min_reliability_update_ref=70.0,
    max_cloud_update_ref=20.0,
    first_scene_mode="first",
    composite_window_days=30,
    iteration=5,
    output_path = None
)

### (OPTIONAL) Create Animations of Non-registered and Co-registered Data Cubes 

In [5]:
from stac2cube import generate_animation
import xarray as xr

In [ ]:
##############################################################
# Change parameters as needed

display_mode = "rgb" # or ndvi or ndwi

path_non_registered = "./results/test.nc"
path_co_registered = "./results/test_cr.nc"

output_non_reg = "./animations/test.gif"
output_co_reg = "./animations/test_cr.gif"

##############################################################

with xr.open_dataset(path_non_registered) as ds:
    non_registered = ds["Spectral_Temporal_Stack"].load()

with xr.open_dataset(path_co_registered) as ds:
    co_registered = ds["Spectral_Temporal_Stack"].load()

generate_animation(stac=non_registered, output_path=output_non_reg, display_mode=display_mode, frame_interval_ms=200)
generate_animation(stac=co_registered, output_path=output_co_reg, display_mode=display_mode, frame_interval_ms=200)

## 2. Custom ROI Co-Registration

In [ ]:
# Select your polygon on the interactive map and continue with the next cell

import leafmap
import numpy as np
import xarray as xr

with xr.open_dataset("./results/test.nc") as ds:
    stac = ds["Spectral_Temporal_Stack"].load()

xmin, ymin, xmax, ymax = map(float, np.asarray(stac.bbox))

m = leafmap.Map(height="800px")
m.add_basemap("Esri.WorldImagery")

# leafmap expects bounds as [[south, west], [north, east]] i.e. [[ymin, xmin], [ymax, xmax]]
m.fit_bounds([[ymin, xmin], [ymax, xmax]])

m

In [ ]:
from stac2cube import coregister_cube_roi
# roi (leafmap)
polygon_map = m.user_roi["geometry"]
out_ds = coregister_cube_roi(
    input_path= stac,
    roi= polygon_map,
    max_cc= 5,
    time_period= None, #["2024-04-19", "2024-10-30"]
    min_reliability_keep= 10.0,
    min_reliability_update_ref= 40.0,
    max_cloud_update_ref= 20.0,
    first_scene_mode="first",
    composite_window_days=30,
    iteration=5,
    output_path= "./results/roi_coregistration.nc",
)

## 3. Spectral Profile Generator

> **Note:** <br><br>
> You can find zoom tools on the top of the map to navigate.

In [ ]:
from stac2cube import ndvi_click_explorer_plotly

before_path = "./results/test.nc"
after_path  = "./results/test_cr.nc" 

ndvi_click_explorer_plotly(before_path, after_path, rgb_time="median");
